In [ ]:
from sklearn.datasets import load_boston
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import math
from prettytable import PrettyTable
np.set_printoptions(suppress=True, formatter={'float_kind':'{:f}'.format})

In [ ]:
boston = load_boston()

In [ ]:
print(boston.data.shape)

In [ ]:
features = boston.feature_names

In [ ]:
print(boston.DESCR)

In [ ]:
bos = pd.DataFrame(boston.data)
print(bos.head())

In [ ]:
bos.fillna(bos.mean(), inplace=True)

In [ ]:
bos['PRICE'] = boston.target

In [ ]:
Y = bos['PRICE']

In [ ]:
Y.fillna(Y.mean(), inplace=True)

In [ ]:
bos.head()

In [ ]:
del bos['PRICE']

In [ ]:
bos.head()

In [ ]:
X=bos

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.33, random_state = 5)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

In [ ]:
s=StandardScaler()
train_data=s.fit_transform(np.array(X_train))
test_data=s.transform(np.array(X_test))

In [ ]:
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)

In [ ]:
X_train['CRIM'] = X_train[0]
X_train['ZN'] = X_train[1]
X_train['INDUS'] = X_train[2]
X_train['CHAS'] = X_train[3]
X_train['NOX'] = X_train[4]
X_train['RM'] = X_train[5]
X_train['AGE'] = X_train[6]
X_train['DIS'] = X_train[7]
X_train['RAD'] = X_train[8]
X_train['TAX'] = X_train[9]
X_train['PTRATIO'] = X_train[10]
X_train['B'] = X_train[11]
X_train['LSTAT'] = X_train[12]

In [ ]:
X_test['CRIM'] = X_test[0]
X_test['ZN'] = X_test[1]
X_test['INDUS'] = X_test[2]
X_test['CHAS'] = X_test[3]
X_test['NOY'] = X_test[4]
X_test['RM'] = X_test[5]
X_test['AGE'] = X_test[6]
X_test['DIS'] = X_test[7]
X_test['RAD'] = X_test[8]
X_test['TAY'] = X_test[9]
X_test['PTRATIO'] = X_test[10]
X_test['B'] = X_test[11]
X_test['LSTAT'] = X_test[12]

In [ ]:
cols = list()
for i in range(13):
    cols.append(i)
X_train.drop(cols, inplace=True, axis=1)
X_test.drop(cols, inplace=True, axis=1)

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
with open('boston.pickle', 'wb') as f:
    pickle.dump([X_train, Y_train, X_test, Y_test], f)

In [ ]:
X_train.head()

In [ ]:
X_train = np.array(X_train)
Y_train = np.array(Y_train)
X_test = np.array(X_test)
Y_test = np.array(Y_test)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

# Custom SGD

In [ ]:
iterations = 10000
N = len(X_train)
w0 = [0]*13
alpha=-0.0000004
x = X_train
y = Y_train
b0=0
for m in range(iterations):
    for i in range(N):
        w0 = w0-alpha*2*x[i]*(y[i]-(np.dot(np.transpose(w0), x[i])+b0))
        b0 = b0-alpha*2*(y[i]-(np.dot(np.transpose(w0), x[i])+b0))

In [ ]:
print(w0)

In [ ]:
print(b0)

In [ ]:
# First method predictions for test
Y_test_predict_method1 = []
for i in range(len(Y_test)):
    Y_test_predict_method1.append(np.dot(np.transpose(X_test[i]), w0)+b0)

In [ ]:
mse = mean_squared_error(Y_test,Y_test_predict_method1)
mse_str = "{:f}".format(float(mse))
print(mse_str)


# Plots of predicted values Vs actual values of custom SGD

In [ ]:
actual = Y_test
predicted = Y_test_predict_method1
plt.scatter(actual, predicted, label = 'Actual vs. Predicted Y')
plt.title('Actual Y Test and Predicted Y Test')
plt.xlabel('Actual Y')
plt.ylabel('Predicted Y')
plt.grid(b=True, linewidth=0.5)
plt.legend()
plt.show()

# SGDRegressor of sklearn

In [ ]:
def sklearn_SGD(lr_rate_variation, eta0=1, n_iter=10000000):
    clf=SGDRegressor(penalty=None, learning_rate=lr_rate_variation, eta0=eta0, max_iter=n_iter, tol=None)
    clf.fit(X_train, Y_train)
    Y_predict = clf.predict(X_test)
    
    #scatter plot
    plt.scatter(Y_test, Y_predict, label = 'Actual vs. Predicted Y')
    plt.title('Actual Y Test and Predicted Y Test')
    plt.xlabel('Actual Y')
    plt.ylabel('Predicted Y')
    plt.grid(b=True, linewidth=0.5)
    plt.legend()
    plt.show()
    
    sgd_sklearn_error=mean_squared_error(Y_test, Y_predict)
    sgd_sklearn_error_str = "{:f}".format(float(sgd_sklearn_error))
    print('mean sq error=', sgd_sklearn_error_str)
    print('number of iteration=', n_iter)
    
    return (clf.coef_, clf.intercept_, sgd_sklearn_error_str)

In [ ]:
weight_sklearn_sgd, bias_sklearn_sgd, sgd_sklearn_error = sklearn_SGD(lr_rate_variation='constant', \
                                                                      eta0=0.00000001)

In [ ]:
print("Weight" + str(weight_sklearn_sgd))
print("*"*70)
print("Bias" + str(bias_sklearn_sgd))

# Comparison of weights between custom and sklearn SGD

In [ ]:
x = PrettyTable()
x.field_names = ['Features','Custom SGD Weight', 'sklearn SGD Weight']
for i in range(13):
    x.add_row([features[i], w0[i], weight_sklearn_sgd[i]])
print(x)

# Comparison between mean squared error between custom and sklearn SGD

In [ ]:
print("Custom SGD Mean Squared Error= " + str(mse_str))
print("sklearn SGD Mean Squared Error= " + str(sgd_sklearn_error))

# Conclusion

There is slight difference between weights of Custom SGD vs. sklearn's SGD. Mean squared error of Custom SGD turned out to be 33.962077 whereas that of sklearn's SGD is 31.688433.